# Query-based ListNet model selection

This notebook runs RankLib/ListNet model selection for the query-based experiments. Each dataset section loads a deterministic train/validation/test split, loads precomputed query-similarity groups, sweeps learning-rate and epoch settings, and writes a compact JSON result file for later comparison.


Notebook setup. Because this file now lives in `experiments/query_based/`, the repository root is two directories above the notebook.


In [ ]:
%reload_ext autoreload
%autoreload 2
import sys
sys.path.append("../../")

Import dataset loaders, query-group loading, and the ListNet helpers that wrap RankLib training and result aggregation.


In [ ]:
import os
import tempfile
from pathlib import Path

from ltr_utility.dataset import load_by_query_dataset, load_query_similarity, DatasetName
from listnet.listnet_model_selection import listnet_model_selection, make_listnet_result

`base_path` points to the shared datasets directory from the new notebook location. RankLib needs a writable scratch directory: `CACHE` defaults to the platform temp dir, and can be pointed at a RAM disk (e.g. `/dev/shm` on Linux) by setting the `LTR_CACHE` environment variable. The grid and the holdout split are defined once and reused by every dataset section below.


In [ ]:
base_path = Path("../../datasets")
RANKLIB = Path("../../listnet/ranklib.jar")
HOLD_OUT = (0.5, 0.2, 0.3)
LR_VALUES = [0.01, 0.001]
EPOCH_VALUES = [50, 100, 300]

# RankLib scratch space. /dev/shm only exists on Linux, so fall back to the
# platform temp dir; override with LTR_CACHE to point at a RAM disk.
CACHE = Path(os.environ.get("LTR_CACHE", tempfile.gettempdir()))

# FINDHR

FindHR baseline run. The cell loads the split, reads the FindHR query groups, runs the ListNet hyperparameter sweep, and stores the aggregated query-model-selection result.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.FINDHR, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("FINDHR/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("FINDHR/results/listnet_query_model_selection.json")

# FINDHR LIST

FindHR listwise-target variant. It uses the same ListNet sweep but writes results under the `FINDHRLIST` experiment directory.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.FINDHRLIST, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("FINDHR/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("FINDHRLIST/results/listnet_query_model_selection.json")

# MQ2007

MQ2007 run using precomputed MQ2007 query groups. The output JSON records the best ListNet settings and scores for this query-based setup.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.MQ, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("MQ2007/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("MQ2007/results/listnet_query_model_selection.json")

# MQ2008

MQ2008 run with the same holdout policy and ListNet search space, using the smaller MQ2008 query-group file.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.MQ2008, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("MQ2008/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("MQ2008/results/listnet_query_model_selection.json")

# MQ 2007 LIST

Listwise MQ2007 run. `max_item=100` limits the per-query item count before RankLib training to keep the experiment tractable.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.MQ2007LIST, max_item=100, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("MQ2007LIST/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("MQ2007LIST/results/listnet_query_model_selection.json")

# MQ 2008 LIST

Listwise MQ2008 run with the same capped item count and query-group based model-selection workflow.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.MQ2008LIST, max_item=100, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("MQ2008LIST/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("MQ2008LIST/results/listnet_query_model_selection.json")

# WEB

WEB10K run. The query groups come from the WEB experiment directory and the final JSON is saved next to the other WEB query-based results.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.WEB10, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("WEB/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("WEB/results/listnet_query_model_selection.json")

# YAHOO

Yahoo run. This is the same RankLib/ListNet sweep applied to the Yahoo split and its query-similarity groups.


In [ ]:
train, valid, test, _ = load_by_query_dataset(base_path, DatasetName.YAHOO, hold_out=HOLD_OUT)
query_groups = load_query_similarity(Path("YAHOO/results"))

results = listnet_model_selection(
    cache_path = CACHE,
    rankLib_path = RANKLIB,
    query_groups = query_groups,
    train = train,
    valid = valid,
    test = test,
    lr_values = LR_VALUES,
    epoch_values = EPOCH_VALUES
)
make_listnet_result(results).to_json("YAHOO/results/listnet_query_model_selection.json")